# Simulate GGOR for nparcels using MODFLOW.

* The parcels (area) data are in a shape file.
* The attributes are in the dbf
* The .dbd file is read into pd.DataFrame.
* The meteo is read from existing file or obtained from KNMI site.
* Modflow is used to simultaneously simulate all the parcels dynamically.
* The results are shown for selected parcels (hds, GXG)
* The running water budget is shown for all the parcels combined.

Different scenarios can be dealt with as cases. Scenarios are used to simulate
a regular case or for testing the behavior of the model in given test-circumstances.

A scenario for a test has for instance max 5 parcels. The the number of head
time series to be plotted for verification is then also limited to 5.
Hence a DataFrame needs to be generated from the data that define the properties of these parcels.
This DataFrame can be read from an Excel workbook that bears the name of the case, which allows full control over the parcel properties of the test case.

The next data for a case is the meteo. These data too can be read from
an excel sheet from the same workbook. The required fields are in the
example workbook named "test_basic.xlsx".

@ TO 2020-09-06

The GGOR has been converted to mf6 but using the structured grid, which makes best sense
because we simulate each parcel as a single row of cells in the grid.

Below, the data for each modflow module to be adapted from the defaults, i.e. from those
in the Excel workbook are specified below. Before saving the Modflow files and running
Modflow 6, they well be used to update the default values for each module.

@ TO 2025-07-02
"""


# GGOR run

GGOR sets up the input for Modflow runs it and afterwards its results are analyzed.

To achieve this, `run.py` first loads `mf_setup.y` which loads a workbook in `<case>/data/<casename>.xlsx` which specifies which Modflow packages will be used, and what the values for parameters for all packages are that `flopy` feeds to the different models for which it will generate input files for Modflow. These are the defaults, but can be adjusted so that they are specific for the case. To this end, these parameters enter a large dictioary of which the keys are `Gwf???` (`Gwf`=Ground Water Flow) and `???` stands for a 3-character acronym of each of the Modflow packages (`wel`, `ghb`, `chd` etc), hence, we have keys like `Gwfdis`, `Gwfwel`, `Gwfghb` etc. At then end of `mf_setup` flopy will use this dictionary with parameters to insantionate each package before it generates the input files for Modflow. 

`mf6lab/src/mf_setup`, imports local `<case>/src/mf_apapt` to specify adapt the parameters for the used packages to their actual case-specific values before flopy generates the input files for Modflow. `mf_adapt` imports from local `settings.py` mainly to keep `mf_apdapt` uncluttered and relatively short with least distraction.

Usually the only thing that needs to be done for a new Modflow case can be run, is adapting `mf_adapt` and to some extent `settings.py`.

The outcomes of the Modflow run will have to be analyzed and presented. This is the task of `mf_analyze.py`, which will be run separately after Modflow finished successfully.

Below mf_adapt for thec case AAN_GZK (Gooise Zomer Kade) is broken down in coherent chunks that are individually explained. The real mf_adapt.py is a single module. But one can also run this notebook cell by cell.

@TO 2025-11-18

# Imports, bootstrap, logger, timer

The imports have some specialties that require some attention:

* mf6_bootstrap  It sets adds the mf6lab/src and mf6lab/Projects/<project>/src and the mf6lab/Projects/<project>/cases/<case>/src directories to sys.path to make them importable. It as an activate function, but that is run as a side effect, so nothing but importing mf6_bootstrap needs to be done.
* logging_setup Similarly, it has as side effect that the logging.Basic.configuration is set.
* time is used for timing the functions if not done with timing log_timed.
* log_timed allows timing pieces of code in a with-context, very convenient.
* Several items are imported from local settings.py. Settings.py is to prevent clutter and distracting details inside mf_adapt.py.


In [ ]:
# %% --- imports
import os
import sys
import numpy as np
import time
from pathlib import Path
import logging
from mf6_bootstrap import activate # noqa: F401

activate(__file__) # noqa: RUF100
os.chdir(Path(__file__).parent) # noqa: RUF100

# --- Imports are not missing after activate(__file__)
# ruff: noqa: E402
from settings import props      # pyright: ignore[reportMissingImports]
from timing import log_timed    # pyright: ignore[reportMissingImports]
from mf6tools import Dirs       # pyright: ignore[reportMissingImports]
import ggor_tools as ggt        # pyright: ignore[reportMissingImports]
from logging_setup import configure_logging # pyright: ignore[reportMissingImports]

# --- setting up the logger
logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)
configure_logging()

# --- Rest of the script
case_name = Path(__file__).parent.parent.parts[-1]

In [ ]:
# --- set start time to allow measuring duration of entire script
start_script = time.perf_counter()

case_name = Path(__file__).parent.parent.parts[-1]

logger.info("Running module as a script")

dirs = Dirs()
dirs.meteo = os.path.join(Path(dirs.proj).parent, 'data', 'meteo')
dirs.bofek = os.path.join(Path(dirs.proj).parent, 'data', 'bofek')

# Setting data for Modflow packages

First get the time (meteo) data, add columns for hydrological year and hand-measurements (14th and 28th of each month) to later compute GXG, for summer to allow changeing ditch levels at beginning and end of summer (Instead of a column in the tdata DataFrame, we could also use the timestamp in tdata.index).

Next get the data pertaining to the parcels. It reads the shape file with all its data. Data missing in the shapefile that are still needed in the compuations are added to the columns of the parcel_data GeoDataFrame. The defaults are taken from ggt (GGOR/src/ggortools.py). A min and a max value for the possible parcel width are also set.

In [ ]:

# %% --- tdis ======  time discretization
with log_timed(logger, 'tdata generated and pickled'):
    tdata = ggt.get_tdata(dirs=dirs, stn=240,
                  start='20100101', end='20191231',
                  folder=dirs.data_meteo)
    tdata_file = os.path.join(dirs.data, 'tdata.pkl')
    tdata.to_pickle(tdata_file)
    
with log_timed(logger, "Parcel_data generated and pickled"):
    parcel_data = ggt.get_parcel_data(dirs=dirs, defaults=ggt.defaults, BMINMAX=(5, 250))
    pdata_file = os.path.join(dirs.data, 'parcel_data.pkl')
    parcel_data.to_pickle(pdata_file)

# Simulation time and stress periods for Modflow

The times and timestep are extracted form tdata.index. The stress periods will coinside with days defined by tdata.index, which has one value per day. 10 years implies 3652 or os days and stress periods.

In [ ]:

start_date_time = str(tdata.index[0])

nper, nstep, tsmult = len(tdata), 1, 1.0

# --- Because we change boundary conditions, nper > 0
dt = np.diff(tdata.index - tdata.index[0]) / np.timedelta64(1, 'D')
dt = np.hstack((dt[0], dt)) # Assume dt[0] ==  t[1] - t[0])

period_data = [[sp_time, nstep, tsmult] for sp_time in dt]

Simtdis = {'perioddata': period_data,
           'nper': len(period_data),
           'start_date_time': start_date_time,
           'time_units': props['time_units'],
           }


# The Modflow grid

* The number of layers will be 2 (shallow and regional).
* The number of rows will be equal to the number of parcels.
* The number of columns is automatic, its defined by the half-widht of the widest parcel and the default cell width, which is dx=1 m.

The resulting gr is a tools/fdm/src/mfgrid.Grid object. It carries all it needs to define the grid and can be requested for anything that one may want to know about the grid and its cells. Extremely useful when preparing and analyzing Modflow with structured network. So gr is not the grid, but knows everything about the grid and can answer questions about it.

Notice that Modflow 6 is used but with columns rows and layers as in the previous modflow versions. It uses the structured grid as defined by the .dis package.

IDOMAIN is typically Modflow 6. We use it to definen which cells are inactive. Each row of the grid represents a half-cross section of a parcel. Because the width of the model matches the half-width of the widest parcel, the grid is too wide for all other parcels. To accomplish this, the cells of each row beyond the half width of the represented parcel are made inactive.

To make sure each row (parcel) is independent of its neighbors, the conductivity along the columns is set to almost zero (Modflow does not allow zero), i.e. to k22=1e-20.

Further notice the dictionaries named Gwf??? with Gdf standing for Groundwater Flow Model and ??? for the 3-character indication of a Modflow 6 packages. Gwfdis is the dis package, Gsfsto the storage package and so on. Each dictionary contains part of the prefilled default dictionary of each package. This prefilled dictionary will be simpliupdated with the ones specified below and then fed into the call that instantiates each package later on druring in `mf6lab/src/mmf_setup.py` that will be run to generate the input files for Modflow. For instance, we instantiate the dis package of the groundwater flow model in mf6lab/src/mf_setup.py as follows:

In [ ]:

# # --- in mf6lab/src/mf_setup.py:
# 
#     ### Gwfdis ==================
#         if 'Gwfdis' in use_packages:
#             logging.info('Gwfdis')
# 
#             # --- here is the update of the prefilled dictionary at key 'Gwfdis'
#             model_dict['Gwfdis'].update(**mf_adapt.Gwfdis)
# 
#             gr = model_dict['Gwfdis'].pop('gr')
#             
#             # --- Updatig further with values from the network (gr)
#             model_dict['Gwfdis'].update(nlay=gr.nlay, nrow=gr.nrow, ncol=gr.ncol,
#                                         delr=gr.dx, delc=gr.dy,
#                                         top=gr.Z[0], botm=gr.Z[1:]                              
#             )
# 
#            # Then intantiate the Gwfdis packages
#            fp_packages['Gwfdis'] = flopy.mf6.ModflowGwfdis(gwf, **model_dict['Gwfdis'])


In [ ]:

with log_timed(logger, "grid generated"):
    gr = ggt.grid_from_parcel_data(parcel_data=parcel_data,
                                   dx=props['dx'])

# %% --- Gwfdis ======
IDOMAIN = gr.const(1, dtype=int)
for iy, b in zip(range(gr.ny), parcel_data['b']):
    IDOMAIN[:, iy, gr.Xm[iy] > b] = 0

IDOMAIN[gr.DZ < props['minDz']] = -1 # Don't need this in GGOR

Gwfdis = {'gr': gr,
     'idomain': IDOMAIN,
     'length_units': props['length_units']}

# Transient simulation

Modflow will compute transient flow only when the storage coefficiets are defined. 

In [ ]:

# %% --- Gwfsto ======= Storage coefficients

S = ggt.set3D(parcel_data[['sy',  'S2']], gr.shape)
Sy = ggt.set3D(parcel_data[['sy', 'sy']], gr.shape)

Gwfsto = {    
    'ss':  S,
    'sy': Sy,
    'iconvert': props['icelltype'],
    'storagecoefficient': True, # Interpret as S instead of Ss where
    }


# Cell propeties (conductivities)

We already had `ss` and `sy` above. Here we define the conductivities. Notice that `k22` is set to `1e-20`

In [ ]:
# %% --- Gwfnpf ===== Cell properties
parcel_data['kc'] = parcel_data['D_CB'] / parcel_data['c_CB']

Gwfnpf = {
    'k': ggt.set3D(parcel_data[['kh', 'kh2']], gr.shape),
    'k22': 1.e-20,                  # no flow along y=axis
    'k33': ggt.set3D(parcel_data[['kv', 'kv2']], gr.shape),
    'icelltype': props['icelltype'],
}

# Initial heads

The intial heads are set to winter levels. We don't want to start with a dry parcel, mainly to prevent converging issues. The first year of the simulation will be used as run-in, so this choice does not affect the outcomes.

In [ ]:

# %% --- Gwfic ===== Initial heads
Gwfic = {
    'strt': ggt.set3D(parcel_data['h_winter'], gr.shape)
}

# Recharge

The recharge is uniform over the model, while day-aveage values are taken from the column 'RH' in the tdata DataFrame. 'RH' is used by KNMI to indicated the precipitation column in their files.

In [ ]:

# %% --- Gwfrcha ===== Recharge   
rch_spd = {isp: tdata['RH'].iloc[isp] for isp in range(len(tdata))}

Gwfrcha = {  
    'recharge': rch_spd,
    'readasarrays': True,
    'print_input': True,    
}


# Evaporation and transpiration

Modflow needs a 'surface' above with the evaporation is not reduced by lack of soil moisture. The AHN (ground-surface) elevation of the parcel is used minus an parcel_specific depth 'ET_surfd'. Modflow also needs the extinction depth. A water table below the extinction elevation reduces evaporation/transpiration to zero. Between the surface and `surface - Exdp` potential evaporation is linearly reduced to from full to zero.

In [ ]:

# %% --- Gwfevta ================================
with log_timed(logger, "Gwfevta gnerated"):
    Gwfevta = {
        'readasarrays': True,
        'ievt': None,
        'surface': {0: ggt.set3D(parcel_data['AHN'] - parcel_data['ET_surfd'], shape=gr.shape)[0]},
        'depth': {0: ggt.set3D(parcel_data['ET_exdp'], shape=gr.shape)[0]},
        'rate': {isp: tdata['EV24'].iloc[isp] for isp in range(len(tdata))}
    }


# Boundary conditions

These are settings that determine the bi-directional exchange of water between the model and the outside world. These conditions are WEL (fixe flows in designated cells), CHD (fixed head in designated cells, which will cause exchange of water with the outside world), GHB (general head boundaries), RIV (rivers), DRN (drians). GHB relate designated cells to the outside world through a conductance (inverse resistance) allowing 2-way flow depending on the head outside, the head in the cell and its conductance. RIV does the same, however when the head sinks below the river bottom, the infiltration is constant, indipendent of the groundwater head. DRN allow water to flow out when the head is above the drain's elevation, but water cannot enter the model when the head is below the drain's elevation. Drains are useful to simulate runoff when put at ground surface and actual tile drains when put on the elevation of these tile drains.

The behavior of the RIV cells can be made the same as that of the DRN cells by setting the river stage (water level) equal to bottom elevation. Then when the head sinks below the bottom, not infiltration can occur. RIV are used in the GGOR to make outflow from the ground into the ditch easier than inflow. When the water level in the ditch is higher than that of the adjacent groundwater, infiltration will only go through the constant conductance of the GHB. Then the head of the groundwater next to the ditch is higher than the water level in the ditch, then the outflow will be through the conductance of the GHB-cells plus that through the conductance of the RIV-cells, hence with lower resistance.   

In [ ]:

# %% --- Prepare boundary conditions ==========
mon = np.array([dt.month for dt in tdata.index])
day = np.array([dt.day for dt in tdata.index])

Isp_start_summer = np.where(np.logical_and(mon ==  4, day == 1))[0]
Isp_start_winter = np.where(np.logical_and(mon == 10, day == 1))[0]


# Fixed flows (using WEL package)

Used to define fixed seepage from the regioinal aquifer upward (seldom downward)

In [ ]:

# %% --- wel ===== used to model given seepage from regional aquifer
active = IDOMAIN > 0
Iwel = gr.NOD[-1][active[-1]]

Q = (parcel_data['q_up'].values[:, np.newaxis] * gr.Area) [np.newaxis, : ,:] * gr.const(1)
Q[:-1, : ,:] = 0.0

spd = ([(lrc, Qw) for lrc, Qw in 
            zip(gr.lrc_from_iglob(Iwel,  astuples=True), Q.ravel()[Iwel])])

# Only the first SP needs data, as long as seepage is constant.
# TODO: this likely changes in the future, to monthly seepage values.
# TODO: Specify how monthly seepage values for all parcels are imported.
# TODO: This import may require an extended data file originating from a regional model.
# TODO: its dimensions should be (nparcel x months in time series)
stress_period_data = {0: spd}

Gwfwel = {
    'stress_period_data': stress_period_data,
    'maxbound': len(Iwel)
}    

# General head boundaries (GHB)

Use for exchange between ditch and ground, both ways.
The external heads (ditch levels) will fluctuate between summer and winter level

In [ ]:

# %% --- Gwfghb ===== Is used for flow from and toward ditches
    
# --- First cell of top and bottom layer (always). Conduction of bottom layer may be zero.
Ighb = gr.NOD[[0, -1], :, 0].flatten()

LRC_ghb = gr.lrc_from_iglob(Ighb, astuples=True)

# --- Get conductance for the GHB (connection soil Ditch in top and bottom layer)
# --- i.e. one value per parcel in the top layer and one value in the bottom layer.

# --- Use analytic ditch resistance with layer thickness and no partial penetration
condGHB = ggt.get_cond_GHB(pdata=parcel_data, gr=gr)

# --- Get the GHB heads for summer and winter for these ditches
hw = np.vstack((parcel_data['h_winter'], parcel_data['h_winter']))
hs = np.vstack((parcel_data['h_summer'], parcel_data['h_summer']))

ghb_winter = [(lrc, head, cond) for lrc, head, cond in
                        zip(LRC_ghb, hw.ravel(), condGHB.ravel())]
ghb_summer = [(lrc, head, cond) for lrc, head, cond in
                        zip(LRC_ghb, hs.ravel(), condGHB.ravel())]

# --- Input for the first stress period
stress_period_data = {0: ghb_summer if tdata.iloc[0]['summer'] is True else ghb_winter}

# --- Only generate input when summer changes to winter or vice versa
for isp in Isp_start_summer:
    stress_period_data[isp] = ghb_summer
for isp in Isp_start_winter:
    stress_period_data[isp] = ghb_winter

Gwfghb = {
    'stress_period_data': stress_period_data,
    'maxbound': len(LRC_ghb),
}


# River boundary conditions (RIV)

The RIV boundary is used to reduce the resistance between ground and adjacent ditch. Ir works such that only flow from the groun into the ditch is possible. This is achieved by setting the ditch bottom equal to the ditch stage at each time step (winter / summer).

In [ ]:

# %% --- Gwfriv ==============Is used for extra flow toward ditch (lower flow to than from)
# Riv cells are the same as GHB cells
LRC_riv = LRC_ghb

condRIV = ggt.get_RIV_Cond(pdata=parcel_data, gr=gr)

riv_winter = [(lic, stage, cond, rbot) for lic, stage, cond, rbot in
                            zip(LRC_riv, hw.ravel(), condRIV.ravel(), hw.ravel())]
riv_summer = [(lic, stage, cond, rbot) for lic, stage, cond, rbot in
                        zip(LRC_riv, hs.ravel(), condRIV.ravel(), hs.ravel())]

# --- Input for the first stress period
stress_period_data = {0: riv_summer if tdata.iloc[0]['summer'] is True else riv_winter}

# --- Only generates input when summer changes to winter and vice versa.
for isp in Isp_start_summer:
    stress_period_data[isp] = riv_summer
for isp in Isp_start_winter:
    stress_period_data[isp] = riv_winter

Gwfriv = {
    'stress_period_data': stress_period_data,
    'maxbound': len(LRC_riv),
}


# Drain boundary condition (DRN)

The DRN boundary can be used to compute surface runoff. This is done by setting the DRN elevation equal to ground surface. When there are trenches, then the drains at these trenches will get a lower elevation to simulate their runoff. In case there are actual tile drains, the drain elevation will be set to that of the tile drains. Drain distance could then be made equal to that of the drains in the real situation. However, drains will usually lie perpendicular to the long sides of the parcel, which would along the cross section of our GGOR model with one row of cells representing a single parcel. This cannot be modeled within the GGOR directly, but we can still simulate the drains by setting their elevation equal to that of the real-world tile drains and use a parcl-wide representative resistance/conductance, so that the results will be close-enough to those of the real-situation. These results then coincide with the evarage head between the drains in the field.

In [ ]:

# %% --- Gwfdrn ============== Drn is used for drains, surface runoff and trenches
Idrn = gr.NOD[0, :, 1:][active[0, :, 1:]]
LRC_drn = gr.lrc_from_iglob(Idrn, astuples=True)

# --- Get drain elevation and drain conductance as Ny * Nx array (top layer)
elevation = ggt.get_drain_elev_with_trenches(pdata=parcel_data, gr=gr, d_drn=props['drain_depth']).ravel()[Idrn]
condDRN   = ggt.get_cond_DRN(pdata=parcel_data, gr=gr).ravel()[Idrn]

# --- Stress period data
#dtype = flopy.modflow.ModflowDrn.get_default_dtype()
#spd   = np.recarray(gr.nod, dtype=dtype)

spd = [(lrc, elev, cond) for lrc, elev, cond in zip(LRC_drn, elevation, condDRN)]

# --- Input is required only for the first SP, because drain data are constant.
stress_period_data = {0: spd}

Gwfdrn = {
    'stress_period_data': stress_period_data,
    'maxbound': len(LRC_drn),
}


# Output control

This specifies what output Modflow will produce and how often. The oc-frequency is set in the props variable in the local `settings.py` module.

In [ ]:

# %% --- Gwfoc ==== Output control for flow model
Gwfoc = {'head_filerecord':   os.path.join(dirs.SIM, "{}Gwf.hds".format(sim_name)),
         'budget_filerecord': os.path.join(dirs.SIM, "{}Gwf.cbc".format(sim_name)),
         'saverecord': [("HEAD", "FREQUENCY", props['oc_frequency']),
                        ("BUDGET", "FREQUENCY", props['oc_frequency'])],
}

logger.info(f"mf_adapt finished in {time.perf_counter() - start_script:.2f} seconds")

# Saving tdata and parcel_data for use after Modflow ran

tdata and parcel_data are relative expensive to compute, while they are also necessary to present the results after Modflow has ran. Therefore, they are saved for use in mf_analyze, where columns are added like `GHG`, `GVG` and `GLG`.

In [ ]:

# --- Pickling the parcel_data geopandas.GeoDataFrame
pdata_pkl = os.path.join(dirs.data, 'parcel_data.pkl')
parcel_data.to_pickle(pdata_pkl)
logger.info(f"parcel_data pickled to {pdata_pkl}")

# --- Pickling the tdata pd.DataFrame
tdata_pkl = os.path.join(dirs.data, 'tdata.pkl')
logger.info(f"tdata pickled to {tdata_pkl}")
tdata.to_pickle(tdata_pkl)

if __name__ == '__main__':
    print('---- All done mf_adapt ! ----')
